# 自定义中间件 Node-style hooks

In [1]:
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from typing import Any
from langchain.agents.middleware import after_model, before_model,before_agent,after_agent


@before_model
def before_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content+="-------->before model middleware <---------"
    return None

@after_model
def after_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content+="-------->after model middleware <---------"
    return None

@before_agent
def before_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content+="-------->before agent middleware <---------"
    return None

@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content+="-------->after agent middleware <---------"
    return None

In [2]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from rich import print as rprint


model = ChatOpenAI(
    model="qwen3.5:9b-q4_K_M",
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    temperature=0,
    # 关闭思考模式，直接生成结构化结果。
    reasoning_effort="none",
    timeout=120,
    max_retries=0,
)

agent=create_agent(
    model=model,
    middleware=[before_model_middleware, after_model_middleware, before_agent_middleware, after_agent_middleware], # 添加中间件
)

res=agent.invoke({
    "messages": [
        {"role": "user", "content": "你好"}
    ]
})

for msg in res["messages"]:
    rprint(msg.content)

你好-------->before agent middleware <----------------->before model middleware <---------

你好！你提供的箭头图示似乎是在描述一个典型的 **LLM 应用架构中的中间件（Middleware）执行流程**，常见于 
LangChain、LlamaIndex 或自研框架中。不过你的描述存在一点逻辑顺序问题，我来帮你梳理并修正：

---

### ✅ 正确的典型调用链顺序（以生成式 AI 应用为例）：
```
用户请求 
→ before_agent_middleware (Agent 层前置处理) 
→ agent_logic (Agent 执行规划/工具调用等) 
→ after_agent_middleware (Agent 层后置处理) 
→ before_model_middleware (模型层前置处理，如提示词注入、安全过滤) 
→ model_inference (大模型推理) 
→ after_model_middleware (模型层后置处理，如输出清洗、结构化解析)
```

---

### 🔍 关键说明：
1. **Agent Middleware**  
   - `before_agent`：在 Agent 决策前拦截（例如：检查用户权限、注入系统指令）。
   - `after_agent`：在 Agent 返回结果后处理（例如：格式化最终响应、记录日志）。

2. **Model Middleware**  
   - `before_model`：向模型输入前操作（如添加安全围栏、动态调整 prompt）。
   - `after_model`：模型输出后操作（如敏感词过滤、JSON 解析验证）。

3. **常见误区**  
   ❌ "before model" 应该在 "agent" 之后执行（因为 Agent 可能调用多次模型）  
   ✅ 实际流程中，每个工具调用/思考步骤都会独立经过 `model middleware`

---

### 💡 需要我帮你：
- 画出完整的时序图？
- 提供具体框架（如 LangChain）的代码示例？
- 解释某个中间件的最佳实践？
- 设计自定义中间件的实现方案？

请告诉我你的具体需求方向！ 😊-------->after model middleware <----------------->after agent middleware <---------

# 自定义中间件
## 基于类的实现
before_model 场景：
- 消息修剪
- Pii脱敏
- 输入验证
- 条件路由
after_model 场景：
- 输出格式化
- 输出验证
- 格式化响应
- 统计信息
- 状态更新



In [3]:
from langchain.agents.middleware import AgentMiddleware

class MyMiddleware(AgentMiddleware):
    
    def before_model(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content+="-------->before model middleware <---------"
        return None

    def after_model(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content+="-------->after model middleware <---------"
        return None

    def before_agent(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content+="-------->before agent middleware <---------"
        return None

    def after_agent(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content+="-------->after agent middleware <---------"
        return None

In [4]:
my_middleware = MyMiddleware()

agent=create_agent(
    model=model,
    middleware=[my_middleware], # 添加中间件
)

res=agent.invoke({
    "messages": [
        {"role": "user", "content": "你好"}
    ]
})
for msg in res["messages"]:
    rprint(msg.content)

你好-------->before agent middleware <----------------->before model middleware <---------

你好！你提供的箭头图示似乎是在描述一个典型的 **LLM 应用架构中的中间件（Middleware）执行流程**，常见于 
LangChain、LlamaIndex 或自研框架中。不过你的描述存在一点逻辑顺序问题，我来帮你梳理并修正：

---

### ✅ 正确的典型调用链顺序（以生成式 AI 应用为例）：
```
用户请求 
→ before_agent_middleware (Agent 层前置处理) 
→ agent_logic (Agent 执行规划/工具调用等) 
→ after_agent_middleware (Agent 层后置处理) 
→ before_model_middleware (模型层前置处理，如提示词注入、安全过滤) 
→ model_inference (大模型推理) 
→ after_model_middleware (模型层后置处理，如输出清洗、结构化解析)
```

---

### 🔍 关键说明：
1. **Agent Middleware**  
   - `before_agent`：在 Agent 决策前拦截（例如：检查用户权限、注入系统指令）。
   - `after_agent`：在 Agent 返回结果后处理（例如：格式化最终响应、记录日志）。

2. **Model Middleware**  
   - `before_model`：向模型输入前操作（如添加安全围栏、动态调整 prompt）。
   - `after_model`：模型输出后操作（如敏感词过滤、JSON 解析验证）。

3. **常见误区**  
   ❌ "before model" 应该在 "agent" 之后执行（因为 Agent 可能调用多次模型）  
   ✅ 实际流程中，每个工具调用/思考步骤都会独立经过 `model middleware`

---

### 💡 需要我帮你：
- 画出完整的时序图？
- 提供具体框架（如 LangChain）的代码示例？
- 解释某个中间件的最佳实践？
- 设计自定义中间件的实现方案？

请告诉我你的具体需求方向！ 😊-------->after model middleware <----------------->after agent middleware <---------

before_model 中间件的执行顺序和传递顺序一致
after_model中间件的执行顺序和传递顺序相反
wrap_model_call中间件的执行顺序是：先传递的包在最外层，即洋葱结构